# Code for merging Donor meta-data + assigning groups and parse data

In [1]:
import scanpy as sc

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import datetime
import glob

current_time = f"{datetime.datetime.now():%Y-%m-%d %H:%M}"

/home/frederik/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
donor_info = pd.read_csv('../../Donor_info_v2.csv', sep=';')

Four groups are defined. 
* T1D 
* T2D
* ND - Lean (as BMI < 30)
* ND - Obese (as BMI > 30)

In [ ]:
def group(df):
    if (df['T1D'] == 'yes') & (df['T2D'] == 'no'):
        return 'T1D'
    elif (df['T1D'] == 'no') & (df['T2D'] == 'yes'):
        return 'T2D'
    if (df['T1D'] == 'no') & (df['T2D'] == 'no'):
        if(df['BMI'] < 30):
            return 'ND - Lean'
        elif(df['BMI'] >= 30):
            return 'ND - Obese'
        else: 
            return 'NaN'
    

donor_info['group'] = donor_info.apply(group, axis = 1)

In [9]:
set(donor_info['group']) # Sanity check that all groups have assigned donors

{'ND - Lean', 'ND - Obese', 'NaN', 'T1D', 'T2D'}

In [5]:
cat_order = ['ND - Lean', 'T1D', 'ND - Obese', 'T2D', 'NaN']
donor_info['group'] = donor_info['group'].astype('category').cat.reorder_categories(cat_order)

In [ ]:
# Scanpy creates alot of columns. Only keep relevant
scanpy_columns = ['sample', 'gene_count', 'n_genes_by_counts',
                'log1p_n_genes_by_counts', 'pct_counts_mt',
                'total_counts', 'total_counts_ribo', 'pct_counts_ribo',
                'doublet_score', 'total_counts_hb', 'pct_counts_hb', 'leiden', 'annotation', 'annotation_subclustering_updated']

Add donor-metadata to all_cells object

In [7]:
all_cells = sc.read_h5ad('../../h5ad/parse_snRNA-seq_all_QC_cells.h5ad')

In [8]:
all_cells.obs = pd.merge(all_cells.obs[scanpy_columns], donor_info, left_on='sample', right_on='sample').set_index(all_cells.obs.index)
all_cells.write_h5ad('../../h5ad/parse_snRNA-seq_all_QC_cells.h5ad')

Add donor-metadata to individual subclusters

In [18]:
subcluster_paths = glob.glob('../../h5ad/subcluster*_v2.h5ad')
subcluster_names = [subcluster_path.replace('../../h5ad/subcluster_', '').replace('_v2.h5ad', '') for subcluster_path in subcluster_paths]

subcluster_adata = {}
for i, subcluster_path in enumerate(subcluster_paths):
    subcluster_adata[subcluster_names[i]] = sc.read_h5ad(subcluster_path)

In [20]:
for name, value in subcluster_adata.items():
    subcluster_adata[name].obs = pd.merge(subcluster_adata[name].obs[scanpy_columns], donor_info, left_on='sample', right_on='sample').set_index(subcluster_adata[name].obs.index)
    subcluster_adata[name].write_h5ad(f'../../h5ad/subcluster_{name}_v2.h5ad')